# llm_report

> Phase 10 summaries and `report.md`: welfare per cell, harness benefit per model, the model-size contrast, numpy versus LLM, the equal-budget arm, the prompt-only arms and the liar, and the interface, token and cost table, with the provenance (plan hash, commit, models, prices and their date, cost) first.

In [ ]:
#| default_exp llm_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math

import numpy as np

from explRL.bench_stats import bootCI
from explRL.llm_compare import compareNumpy

NAN = float('nan')


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def _sel(rows, **kw):
    return [r for r in rows if all(str(r.get(k)) == str(v) if not isinstance(v, (int, float)) else _f(r.get(k)) == v
                                   for k, v in kw.items())]


def _stat(x):
    x = np.array([_f(v) for v in x], float)
    x = x[~np.isnan(x)]
    return dict(mean=float(x.mean()) if len(x) else NAN, ci=list(bootCI(x)), n=int(len(x)))


def _paired(a, b, metric):
    "Per-seed differences a - b over the seeds (and episodes) both have."
    A = {(int(_f(r['seed'])), int(_f(r.get('episode', 0)))): _f(r[metric]) for r in a}
    B = {(int(_f(r['seed'])), int(_f(r.get('episode', 0)))): _f(r[metric]) for r in b}
    return [A[k] - B[k] for k in sorted(set(A) & set(B))]


def mainTable(rows, study='main', metric='welfare'):
    "{(task, channel, N): stats of `metric`} for one study."
    out = {}
    for r in _sel(rows, study=study):
        out.setdefault((r['task'], r['channel'], int(_f(r['N']))), []).append(r)
    return {k: dict(_stat(r[metric] for r in v), invalid=_stat(_f(r['invalid']) / max(_f(r['calls']), 1) for r in v)['mean'])
            for k, v in sorted(out.items())}


def harnessBenefit(rows, study='main', metric='welfare'):
    "{(task, channel, N): stats of the channel's welfare minus the 'none' cell's, paired on seed}."
    out = {}
    for (t, ch, N) in mainTable(rows, study, metric):
        if ch != 'none':
            out[(t, ch, N)] = _stat(_paired(_sel(rows, study=study, task=t, channel=ch, N=N),
                                            _sel(rows, study=study, task=t, channel='none', N=N), metric))
    return out


def sizeContrast(rows, sizes=(1, 4), metric='welfare'):
    "Board: per channel, (Sonnet benefit - Haiku benefit) averaged over `sizes`, paired on seed ('shrinks' if CI < 0)."
    out = {}
    for ch in ('blackboard', 'messages', 'hierarchical'):
        per = []
        for N in sizes:
            s = {}
            for st in ('sonnet', 'main'):
                sel = lambda c: {int(_f(r['seed'])): _f(r[metric]) for r in _sel(rows, study=st, task='board', channel=c, N=N)}
                a, b = sel(ch), sel('none')
                s[st] = {k: a[k] - b[k] for k in set(a) & set(b)}
            per.append({k: s['sonnet'][k] - s['main'][k] for k in set(s['sonnet']) & set(s['main'])})
        seeds = sorted(set.intersection(*[set(p) for p in per])) if per else []
        d = [np.mean([p[k] for p in per]) for k in seeds]
        st = _stat(d)
        st['verdict'] = 'shrinks' if st['ci'][1] < 0 else 'grows' if st['ci'][0] > 0 else 'unclear'
        out[ch] = st
    return out


def armTable(rows):
    "Each arm's difference from its control cell, paired on seed (and episode), with bootstrap CIs."
    fr = lambda lv: _sel(rows, study='framing', level=lv)
    out = {f'framing {lv} - own': _stat(_paired(fr(lv), fr('own'), 'welfare')) for lv in ('team', 'credit')}
    out.update({f'framing {lv} - own: {m}': _stat(_paired(fr(lv), fr('own'), m))
                for m in ('exploreShare', 'reportShare', 'freeRiding') for lv in ('team', 'credit')})
    out.update({f'memory {m} - full': _stat(_paired(_sel(rows, study='memory', level=m), fr('own'), 'welfare'))
                for m in ('last8', 'notes')})
    p = lambda lv: [r for r in _sel(rows, study='persist', level=lv) if int(_f(r['episode'])) == 3]
    out['persist lessons - none (episode 4)'] = _stat(_paired(p('lessons=1'), p('lessons=0'), 'welfare'))
    for t in ('forage', 'board'):
        loss = {}
        for ch in ('blackboard', 'messages'):
            lv = lambda n: _sel(rows, study='liar', level=f'{t}|{ch}|liar={n}')
            d = _paired(lv(0), lv(1), 'welfareHonest')
            out[f'liar {t} {ch}: honest loss'] = _stat(d)
            loss[ch] = dict(zip(sorted({int(_f(r['seed'])) for r in lv(0)}), d))
        dd = [loss['blackboard'][k] - loss['messages'][k] for k in sorted(set(loss['blackboard']) & set(loss['messages']))]
        out[f'liar {t}: blackboard - messages loss (H3 sign: > 0)'] = _stat(dd)
    return out


def budgetTable(rows):
    out = {}
    for r in _sel(rows, study='budget'):
        out.setdefault((r['channel'], int(_f(r['N']))), []).append(r['welfare'])
    return {k: _stat(v) for k, v in sorted(out.items())}


def interfaceTable(rows):
    "Per study: runs, calls, invalid and refusal rates, cache hits, prompt-cache reads, tokens and cost."
    out = {}
    for st in sorted({r['study'] for r in rows if r.get('runner') == 'llm'}):
        rs = _sel(rows, study=st, runner='llm')
        tot = lambda k: float(np.nansum([_f(r[k]) for r in rs]))
        c = max(tot('calls'), 1)
        out[st] = dict(runs=len(rs), calls=int(tot('calls')), invalid=tot('invalid') / c, refusals=int(tot('refusals')),
                       maxTokenStops=int(tot('maxTokenStops')),
                       promptCacheRead=float(np.nanmean([_f(r['promptCacheRead']) for r in rs])),
                       inputTokens=int(tot('inputTokens') + tot('cacheWriteTokens') + tot('cacheReadTokens')),
                       outputTokens=int(tot('outputTokens')), cost=tot('cost'))
    return out


def summaries(rows, numpyRows=()):
    llm = [r for r in rows if r.get('runner') == 'llm']
    return dict(main=mainTable(llm), benefit=harnessBenefit(llm), sonnetBenefit=harnessBenefit(llm, 'sonnet'),
                sizeContrast=sizeContrast(llm), numpy=compareNumpy(_sel(llm, study='main'), list(numpyRows))
                if numpyRows else {}, budget=budgetTable(llm), arms=armTable(llm), interface=interfaceTable(llm))


def _fmt(s):
    m, (lo, hi) = s['mean'], s['ci']
    return 'n/a' if math.isnan(m) else f'{m:.3f} [{lo:.3f}, {hi:.3f}]'


def writeReport(path, S, meta):
    "report.md: provenance first (plan hash, commit, models, prices and their date, cost), then the tables."
    L = [f"# Phase 10: language-model agents ({meta['mode']})", '',
         'All results are exploratory (pre-specified in the analysis plan; no confirmatory test). Live reruns differ: '
         'only the response cache makes reruns exact.', '',
         f"- Analysis plan hash: `{meta['planHash']}`; git commit: `{meta['commit']}`",
         f"- Models: {', '.join(meta['models'])}; effort: {meta['effort']}",
         f"- Prices ($ per million tokens, checked {meta['pricesChecked']}): " +
         '; '.join(f"{m}: in {p['input']}, out {p['output']}, cache read {p['cacheRead']}, cache write {p['cacheWrite']}"
                   for m, p in meta['prices'].items()),
         f"- Cost: ${meta['cost']:.2f} for these runs' replies (a reply served from the cache counts at its original cost); "
         f"${meta.get('spent', 0.0):.2f} charged in all (spend ledger) of the ${meta['cap']:.0f} cap; calls {meta['calls']}, cache hits "
         f"{meta['cacheHits']}; wall time {meta['wall']:.0f} s", '',
         '## Welfare per agent-turn (main grid, mean [95% CI])', '', '| task | channel | N | welfare | invalid share |',
         '|---|---|---|---|---|']
    L += [f'| {t} | {ch} | {N} | {_fmt(s)} | {s["invalid"]:.3f} |' for (t, ch, N), s in S['main'].items()]
    L += ['', '## Harness benefit (channel minus none, paired on seed)', '', '| model | task | channel | N | benefit |',
          '|---|---|---|---|---|']
    L += [f'| {m} | {t} | {ch} | {N} | {_fmt(s)} |' for m, key in (('Haiku', 'benefit'), ('Sonnet', 'sonnetBenefit'))
          for (t, ch, N), s in S[key].items()]
    L += ['', '## Model size (board; Sonnet benefit minus Haiku benefit, sizes 1 and 4)', '',
          '| channel | difference | verdict |', '|---|---|---|']
    L += [f'| {ch} | {_fmt(s)} | {s["verdict"]} |' for ch, s in S['sizeContrast'].items()]
    L += ['', '## Numpy predicts LLM? (Spearman rho over the (channel, N) cells)', '', '| task | rho | 95% CI | cells | seeds |',
          '|---|---|---|---|---|']
    L += [f"| {t} | {v['rho']:.2f} | [{v['ci'][0]:.2f}, {v['ci'][1]:.2f}] | {v['cells']} | {v['seeds']} |"
          for t, v in S['numpy'].items()]
    L += ['', '## Equal budget (200 agent-turns per team)', '', '| channel | N | welfare per agent-turn |', '|---|---|---|']
    L += [f'| {ch} | {N} | {_fmt(s)} |' for (ch, N), s in S['budget'].items()]
    L += ['', '## Arms (difference from control, paired)', '', '| contrast | mean [95% CI] | n |', '|---|---|---|']
    L += [f'| {k} | {_fmt(s)} | {s["n"]} |' for k, s in S['arms'].items()]
    L += ['', '## Interface, tokens and cost', '', '| study | runs | calls | invalid | refusals | max_tokens | '
          'prompt-cache read | input tokens | output tokens | cost $ |', '|---|---|---|---|---|---|---|---|---|---|']
    L += [f"| {st} | {v['runs']} | {v['calls']} | {v['invalid']:.3f} | {v['refusals']} | {v['maxTokenStops']} | "
          f"{v['promptCacheRead']:.2f} | {v['inputTokens']} | {v['outputTokens']} | {v['cost']:.2f} |"
          for st, v in S['interface'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path

## Tests

In [ ]:
import math, tempfile
from pathlib import Path
from explRL.llm_report import *
from explRL.bench_grid import runGrid
from explRL.llm_client import PRICES
from explRL.llm_grid import SONNET, HAIKU, planHash, quickGrid, runAnyCell, withClient

rows = runGrid(withClient(quickGrid(), dict(kind='fake')), workers=1, run=runAnyCell)
llm, npy = [r for r in rows if r['runner'] == 'llm'], [r for r in rows if r['runner'] == 'numpy']
S = summaries(llm, npy)
assert len(S['main']) == 2 * 4 * 4 and all(s['n'] == 1 for s in S['main'].values())
assert len(S['benefit']) == 2 * 3 * 4 and set(S['sizeContrast']) == {'blackboard', 'messages', 'hierarchical'}
assert set(S['numpy']) == {'forage', 'board'} and S['numpy']['board']['cells'] == 16
assert S['interface']['main']['calls'] == sum(int(r['calls']) for r in llm if r['study'] == 'main')
assert 'liar forage: blackboard - messages loss (H3 sign: > 0)' in S['arms'] and len(S['budget']) == 8

# paired differences match by seed
a = [dict(seed=1, welfare=3.0), dict(seed=2, welfare=5.0)]
b = [dict(seed=2, welfare=1.0), dict(seed=3, welfare=0.0)]
assert _paired(a, b, 'welfare') == [4.0]

meta = dict(mode='quick', planHash=planHash(), commit='abc1234', models=[HAIKU, SONNET], effort='low',
            pricesChecked=PRICES[HAIKU]['checked'], prices={m: PRICES[m] for m in (HAIKU, SONNET)}, cost=0.0123,
            cap=150.0, calls=10, cacheHits=0, wall=1.0)
p = writeReport(Path(tempfile.mkdtemp()) / 'report.md', S, meta)
txt = p.read_text()
for s in (planHash(), 'abc1234', HAIKU, SONNET, f"checked {PRICES[HAIKU]['checked']}", '$0.01 for these runs', '$0.00 charged in all', '| forage | blackboard | 4 |'):
    assert s in txt, s

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()